In [48]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from catboost import CatBoostClassifier

# ==========================================
# 1. LOAD DATA
# ==========================================

file_name = "Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv"
df = pd.read_csv(file_name)
print(f"Data loaded successfully. Shape: {df.shape}")

# ==========================================
# 2. COORDINATES & DISTANCE CALCULATION
# ==========================================

def haversine(lat1, lon1, lat2, lon2):
    """
    Calculates great-circle distance between two GPS coordinates using Haversine formula.
    """
    R = 6371  # Earth's radius in kilometers
    
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return 0.0
        
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = (
        sin(dlat / 2)**2
        + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    )
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# Distance feature calculation
df["distance_km"] = df.apply(
    lambda x: haversine(
        x["latitude"],
        x["longitude"],
        x["endlatitude"],
        x["endlongitude"]
    ),
    axis=1
)

# Robustly map road_closure to numeric 1 and 0 to force a strong learning signal
if "requires_road_closure" in df.columns:
    df["road_closure"] = df["requires_road_closure"].astype(str).str.strip().str.lower().map({"yes": 1, "true": 1, "1": 1, "no": 0, "false": 0, "0": 0}).fillna(0)
else:
    df["road_closure"] = 0


# ==========================================
# [Screenshot Fix 1] DISTANCE OUTLIERS & LOG SMOOTHING
# ==========================================
# Ensure numeric format
df["distance_km"] = pd.to_numeric(df["distance_km"], errors="coerce")

# Fill missing values
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# Cap extreme outliers (1st to 99th percentile)
q1 = df["distance_km"].quantile(0.01)
q99 = df["distance_km"].quantile(0.99)
df["distance_km"] = df["distance_km"].clip(q1, q99)

# Apply log transformation smoothing
df["distance_km_log"] = np.log1p(df["distance_km"])


# ==========================================
# 3. TARGET MAPPING (Low, High) - Simplified
# ==========================================

priority_map = {
    "Low": 0,
    "High": 1
}

df = df[df["priority"].isin(priority_map.keys())].copy()
df["target"] = df["priority"].map(priority_map)

print("\nTarget Class Distribution:")
print(df["target"].value_counts())

# ==========================================
# 4. TEMPORAL & BONUS INTERACTION FEATURES
# ==========================================

df["start_datetime"] = pd.to_datetime(df["start_datetime"], errors="coerce")
df["resolved_datetime"] = pd.to_datetime(df["resolved_datetime"], errors="coerce")

df["hour"] = df["start_datetime"].dt.hour
df["day_of_week"] = df["start_datetime"].dt.dayofweek
df["month"] = df["start_datetime"].dt.month

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Peak traffic hours mapping
df["is_peak_hour"] = (
    ((df["hour"] >= 8) & (df["hour"] <= 11)) |
    ((df["hour"] >= 17) & (df["hour"] <= 21))
).astype(int)

# Calculate actual duration purely for calculating historical averages (No leakage in training)
df["actual_duration_minutes"] = (df["resolved_datetime"] - df["start_datetime"]).dt.total_seconds() / 60

# ==========================================
# 5. PREVENTING DATA LEAKAGE & THREE-WAY SPLIT
# ==========================================

# 1. First split out test set (20% of total) and temp set (80% of total)
train_val_idx, test_idx = train_test_split(
    df.index, test_size=0.20, stratify=df["target"], random_state=42
)

# 2. Split temp set into train (70% of total) and validation (10% of total)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.125, stratify=df.loc[train_val_idx, "target"], random_state=42
)

train_df = df.loc[train_idx].copy()
val_df = df.loc[val_idx].copy()
test_df = df.loc[test_idx].copy()

# Calculate historical expected duration based on event type on train split only (No Leakage)
historical_duration_map = train_df.groupby("event_type")["actual_duration_minutes"].median().to_dict()
overall_median_duration = train_df["actual_duration_minutes"].median()

# Map the expected duration values to train, validation, and test splits
train_df["expected_duration"] = train_df["event_type"].map(historical_duration_map).fillna(overall_median_duration)
val_df["expected_duration"] = val_df["event_type"].map(historical_duration_map).fillna(overall_median_duration)
test_df["expected_duration"] = test_df["event_type"].map(historical_duration_map).fillna(overall_median_duration)


# ==========================================
# [Screenshot Fix 2 & 5] ROBUST NATIVE CATEGORICAL HANDLING
# ==========================================

categorical_cols = [
    "event_type",
    "event_cause",
    "corridor",
    "zone"
]

# Clean categorical variables and cast strictly to string
for col in categorical_cols:
    train_df[col] = train_df[col].fillna("Unknown").astype(str)
    val_df[col] = val_df[col].fillna("Unknown").astype(str)
    test_df[col] = test_df[col].fillna("Unknown").astype(str)

# Clean, leakage-proof features list
features = [
    "event_type",
    "event_cause",
    "latitude",
    "longitude",
    "distance_km_log",
    "road_closure",    # Active clean 0/1 numeric feature
    "corridor",
    "zone",
    "hour",
    "day_of_week",
    "month",
    "is_peak_hour",
    "expected_duration"
]

X_train = train_df[features].copy()
y_train = train_df["target"]

X_val = val_df[features].copy()
y_val = val_df["target"]

X_test = test_df[features].copy()
y_test = test_df["target"]

# Categoricals to pass natively to CatBoost
cat_features_for_catboost = ["event_type", "event_cause", "corridor", "zone"]

# Impute numeric variables
numeric_cols = [col for col in X_train.columns if col not in cat_features_for_catboost]
for col in numeric_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)


# ==========================================
# [Proven Stable Setup] NO SHRINKAGE CONFIG
# ==========================================

# Balanced weight maps to give strong signal to practical real-world features
feature_weights_map = {f: 1.0 for f in features}
feature_weights_map["corridor"] = 0.05        # Control corridor dominance
feature_weights_map["event_cause"] = 1.8      # Strong weight for Accidents, Floods
feature_weights_map["event_type"] = 1.5       # Strong weight for Event Type
feature_weights_map["road_closure"] = 1.4     # Strong weight for road closure correlation
feature_weights_map["distance_km_log"] = 0.8  # Prevent distance from overwhelming coordinates

model = CatBoostClassifier(
    iterations=800,             # Total trees
    learning_rate=0.03,         # Stable slow learning
    depth=5,                    # Balanced tree depth
    l2_leaf_reg=50,             # Regularization smoothing
    rsm=0.35,                   # Balanced feature subsets per tree split
    random_strength=10,         # High stochastic noise to make tree structures highly diverse
    bootstrap_type="Bayesian",  # Bayesian bootstrap
    bagging_temperature=1.0,    
    feature_weights=feature_weights_map, 
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=42,
    verbose=100,
    auto_class_weights="Balanced"
)

print("\nStarting Stable, Non-Shrinkable CatBoost training...")
# IMPORTANT FIX: Passed use_best_model=False to completely stop early shrinkage to 1 iteration!
model.fit(
    X_train, y_train,
    cat_features=cat_features_for_catboost,
    eval_set=(X_val, y_val),
    use_best_model=False       # This ensures all 800 trees are fully built and utilized!
)
print("Training completed.")

# ==========================================
# 8. PERFORMANCE EVALUATION (Low vs High Report)
# ==========================================

preds = model.predict(X_test)
probs = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, preds)
f1_macro = f1_score(y_test, preds, average="macro")

print("\nPerformance Overview on Untouched Test Set:")
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Macro F1 Score : {f1_macro:.4f}")

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    preds,
    labels=[0, 1],
    target_names=["Low", "High"]
))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, preds))

# ==========================================
# 9. POST-PREDICTION DECISION PIPELINE
# ==========================================

def generate_decision_pipeline(row, pred_class, probabilities):
    """
    Uses non-leaking expected_duration and real-time inputs to calculate
    impact scores, risk levels, resource allocation rules, and routing statuses.
    """
    pred_idx = int(np.array(pred_class).flatten()[0])
    
    priority_weight_map = {
        0: 0.3,  # Low
        1: 1.0   # High
    }
    priority_weight = priority_weight_map.get(pred_idx, 0.3)
    
    confidence = round(max(probabilities) * 100, 2)
    
    duration_val = row.get("expected_duration", 0)
    duration_weight = min(duration_val / 120.0, 1.0)
    
    # road_closure is now numeric (0 or 1)
    road_closure_weight = float(row.get("road_closure", 0.0))
    
    distance_val_log = row.get("distance_km_log", 0.0)
    distance_val = np.expm1(distance_val_log)
    distance_weight = min(distance_val / 10.0, 1.0)
    
    impact_score = (
        (priority_weight * 35) +
        ((confidence / 100.0) * 25) +
        (duration_weight * 20) +
        (road_closure_weight * 10) +
        (distance_weight * 10)
    )
    
    impact_score = int(min(max(impact_score, 0), 100))
    expected_delay = int(impact_score * 0.5 + duration_weight * 20 + distance_weight * 10)
    affected_radius = round((impact_score / 20.0) + (distance_val * 0.1), 1)
    police = int(4 + impact_score * 0.3)
    barricades = max(2, int(impact_score / 8))
    
    if impact_score > 80:
        diversion = "Mandatory"
    elif impact_score > 50:
        diversion = "Recommended"
    else:
        diversion = "No Diversion Required"
        
    if impact_score >= 80:
        risk_level = "Critical"
    elif impact_score >= 60:
        risk_level = "High"
    elif impact_score >= 40:
        risk_level = "Medium"
    else:
        risk_level = "Low"
        
    predicted_priority = "Low"
    if pred_idx == 1:
        predicted_priority = "High"
        
    return {
        "Priority": predicted_priority,
        "Risk Level": risk_level,
        "Confidence": confidence,
        "Impact Score": impact_score,
        "Expected Delay": expected_delay,
        "Affected Radius": affected_radius,
        "Police": police,
        "Barricades": barricades,
        "Diversion": diversion
    }

# ==========================================
# 10. PIPELINE DEMO & OUTPUT VERIFICATION
# ==========================================

print("\n" + "="*50)
print("TEST RUNNING REAL-TIME ROUTING ENGINE")
print("="*50)

sample_index = 0
for idx, true_val in enumerate(y_test):
    if true_val == 1:
        sample_index = idx
        break

sample_row = X_test.iloc[sample_index]
sample_pred = preds[sample_index]
sample_probs = probs[sample_index]

decision_json = generate_decision_pipeline(sample_row, sample_pred, sample_probs)

import json
print(json.dumps(decision_json, indent=4))

# ==========================================
# 11. SAVE COMPLETE BUNDLE & PRINT METRICS
# ==========================================

importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.get_feature_importance()
}).sort_values(by="Importance", ascending=False)

print("\nTop 10 Features (Balanced, Logical & Realistic):")
print(importance_df.head(10))

importance_df.to_csv("feature_importance_catboost.csv", index=False)

model_bundle = {
    "model": model,
    "features": features,
    "categorical_cols": cat_features_for_catboost,
    "historical_duration_map": historical_duration_map,
    "overall_median_duration": overall_median_duration,
    "version": "2.2"
}

with open("catboost_traffic_model.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print("\nBundle saved successfully. Ready to run!")

Data loaded successfully. Shape: (8173, 46)

Target Class Distribution:
target
1    5030
0    3141
Name: count, dtype: int64

Starting Stable, Non-Shrinkable CatBoost training...
0:	learn: 0.9959221	test: 1.0000000	best: 1.0000000 (0)	total: 32.5ms	remaining: 25.9s
100:	learn: 0.9976996	test: 0.9990050	best: 1.0000000 (0)	total: 4.33s	remaining: 30s
200:	learn: 0.9976996	test: 0.9990050	best: 1.0000000 (0)	total: 8.39s	remaining: 25s
300:	learn: 0.9976996	test: 0.9990050	best: 1.0000000 (0)	total: 12.6s	remaining: 20.9s
400:	learn: 0.9976996	test: 0.9990050	best: 1.0000000 (0)	total: 17s	remaining: 17s
500:	learn: 0.9976996	test: 0.9990050	best: 1.0000000 (0)	total: 21.7s	remaining: 13s
600:	learn: 0.9976996	test: 0.9990050	best: 1.0000000 (0)	total: 26.3s	remaining: 8.72s
700:	learn: 0.9976996	test: 0.9990050	best: 1.0000000 (0)	total: 31s	remaining: 4.37s
799:	learn: 0.9981534	test: 0.9990050	best: 1.0000000 (0)	total: 35.4s	remaining: 0us

bestTest = 1
bestIteration = 0

Training co